The first cell is without the plot sentences, but same as second cell

In [ ]:
import os
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp_sparse
import matplotlib.pyplot as plt
import seaborn as sns
import squidpy as sq
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import silhouette_score
from sklearn.metrics import normalized_mutual_info_score
from scipy.stats import entropy as scipy_entropy
import warnings
warnings.filterwarnings('ignore')

OUT_DIR = ("/path/to/data/"
           )
os.makedirs(OUT_DIR, exist_ok=True)

CLUSTER_KEY = 'hc_ward_k10'

# ── Load ─────────────────────────────────────────────────────────────
print("Loading adata...")
adata = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)
print("Loaded:", adata.shape)

X = adata.X.toarray() if sp_sparse.issparse(adata.X) else adata.X

# ── Compositional graph ───────────────────────────────────────────────
print("\nBuilding compositional graph...")
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=10)
A_comp = adata.obsp['connectivities'].copy()
A_comp_norm = A_comp / A_comp.max()
print("Compositional graph built")

# ── Per-slide spatial graph ───────────────────────────────────────────
def build_per_slide_spatial_graph(adata, n_neighbors=15):
    """Build spatial KNN graph independently per slide then combine."""
    n_spots = adata.n_obs
    row_indices = []
    col_indices = []
    data_values = []
    slides = adata.obs['slide_id'].unique()
    print(f"Building spatial graph for {len(slides)} slides...")

    for slide in slides:
        mask = adata.obs['slide_id'] == slide
        idx = np.where(mask)[0]

        if len(idx) < n_neighbors + 1:
            print(f"  Skipping {slide} — too few spots ({len(idx)})")
            continue

        coords_slide = adata.obsm['spatial'][idx]
        A_slide = kneighbors_graph(coords_slide,
                                    n_neighbors=n_neighbors,
                                    mode='connectivity',
                                    include_self=False)
        A_slide = (A_slide + A_slide.T) / 2  # symmetrise

        rows_local, cols_local = A_slide.nonzero()
        rows_global = idx[rows_local]
        cols_global = idx[cols_local]

        row_indices.extend(rows_global.tolist())
        col_indices.extend(cols_global.tolist())
        data_values.extend(A_slide.data.tolist())

    A_spat_full = sp_sparse.csr_matrix(
        (data_values, (row_indices, col_indices)),
        shape=(n_spots, n_spots)
    )
    print(f"Per-slide spatial graph built: {A_spat_full.nnz} edges")
    return A_spat_full

A_spat = build_per_slide_spatial_graph(adata, n_neighbors=15)
A_spat_norm = A_spat / A_spat.max()

# ── Helper functions ──────────────────────────────────────────────────
def compute_homophily_per_slide(adata, label_key, n_neighs=6):
    scores = []
    for slide in adata.obs['slide_id'].unique():
        mask = adata.obs['slide_id'] == slide
        adata_slide = adata[mask].copy()
        if adata_slide.n_obs < 50:
            continue
        sq.gr.spatial_neighbors(adata_slide,
                                 coord_type='generic',
                                 n_neighs=n_neighs,
                                 key_added='spatial_neighbors')
        adj = adata_slide.obsp['spatial_neighbors_connectivities']
        src, tgt = adj.nonzero()
        labels = adata_slide.obs[label_key].values
        same = (labels[src] == labels[tgt]).sum()
        scores.append(same / len(src))
    return np.mean(scores), np.std(scores)

def compute_entropy(adata, label_key, groupby='slide_id'):
    n_groups = adata.obs[groupby].nunique()
    H_max = np.log2(n_groups) if n_groups > 1 else 1
    per_cluster = []
    weights = []
    for cluster in adata.obs[label_key].unique():
        mask = adata.obs[label_key] == cluster
        counts = adata.obs.loc[mask, groupby].value_counts()
        props = counts / counts.sum()
        H = scipy_entropy(props, base=2)
        per_cluster.append(H / H_max)
        weights.append(mask.sum())
    return np.average(per_cluster, weights=weights)

# ── Alpha sweep ───────────────────────────────────────────────────────
alphas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
resolution = 0.5

np.random.seed(42)
n_sample = min(15000, adata.n_obs)
idx_sil = np.random.choice(adata.n_obs, n_sample, replace=False)
X_sil = X[idx_sil]

results = []
print("\nAlpha sweep with all metrics...")
print("="*60)

for alpha in alphas:
    print(f"\nalpha={alpha:.1f}...")
    G_joint = (alpha * A_comp_norm) + ((1 - alpha) * A_spat_norm)

    key = f'joint_alpha_{alpha}'
    sc.tl.leiden(adata, resolution=resolution,
                 adjacency=G_joint, key_added=key,
                 flavor='igraph', n_iterations=2,
                 directed=False, random_state=42)

    n_niches = adata.obs[key].nunique()
    print(f"  n_niches={n_niches}")

    # Silhouette
    labels_sil = adata.obs[key].values[idx_sil]
    sil = silhouette_score(X_sil, labels_sil) if n_niches > 1 else 0
    print(f"  silhouette={sil:.4f}")

    # Homophily per slide
    hom_mean, hom_std = compute_homophily_per_slide(adata, key)
    print(f"  homophily={hom_mean:.4f} ± {hom_std:.4f}")

    # Entropy
    ent_slide = compute_entropy(adata, key, groupby='slide_id')
    ent_patient = compute_entropy(adata, key, groupby='patient')
    print(f"  entropy_slide={ent_slide:.4f}, entropy_patient={ent_patient:.4f}")

    results.append({
        'alpha': alpha,
        'n_niches': n_niches,
        'silhouette': round(sil, 4),
        'homophily_mean': round(hom_mean, 4),
        'homophily_std': round(hom_std, 4),
        'entropy_slide': round(ent_slide, 4),
        'entropy_patient': round(ent_patient, 4),
    })

results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("ALPHA SWEEP COMPLETE")
print("="*60)
print(results_df.to_string(index=False))
results_df.to_csv(os.path.join(OUT_DIR, 'alpha_sweep_results.csv'), index=False)

# ── Plot all four metrics ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(results_df['alpha'], results_df['silhouette'],
                'o-', color='steelblue', linewidth=2)
axes[0, 0].set_xlabel('Alpha')
axes[0, 0].set_ylabel('Silhouette Score')
axes[0, 0].set_title('Compositional compactness'
                      )

axes[0, 1].plot(results_df['alpha'], results_df['homophily_mean'],
                'o-', color='darkorange', linewidth=2)
axes[0, 1].fill_between(
    results_df['alpha'],
    results_df['homophily_mean'] - results_df['homophily_std'],
    results_df['homophily_mean'] + results_df['homophily_std'],
    alpha=0.2, color='darkorange'
)
axes[0, 1].set_xlabel('Alpha')
axes[0, 1].set_ylabel('Spatial Homophily (mean ± std)')
axes[0, 1].set_title('Spatial coherence'
                      )

axes[1, 0].plot(results_df['alpha'], results_df['entropy_slide'],
                'o-', color='forestgreen', linewidth=2, label='By slide')
axes[1, 0].plot(results_df['alpha'], results_df['entropy_patient'],
                's--', color='darkgreen', linewidth=2, label='By patient')
axes[1, 0].set_xlabel('Alpha')
axes[1, 0].set_ylabel('Normalised Entropy')
axes[1, 0].set_title('Cross-slide generalisation'
                      )
axes[1, 0].set_ylim(0, 1)
axes[1, 0].legend()

axes[1, 1].plot(results_df['alpha'], results_df['n_niches'],
                'o-', color='purple', linewidth=2)
axes[1, 1].set_xlabel('Alpha')
axes[1, 1].set_ylabel('Number of niches')
axes[1, 1].set_title('Niche count stability')

plt.suptitle(
    'Joint graph alpha sweep — all metrics\n'
    'α=1.0 = pure composition | α=0.0 = pure spatial',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'alpha_sweep_all_metrics.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: alpha_sweep_all_metrics.png")

In [ ]:
import os
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp_sparse
import matplotlib.pyplot as plt
import seaborn as sns
import squidpy as sq
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import silhouette_score
from sklearn.metrics import normalized_mutual_info_score
from scipy.stats import entropy as scipy_entropy
import warnings
warnings.filterwarnings('ignore')

OUT_DIR = ("/path/to/data/"
           "Cell2location/spatial_mapping/exploration_mean/joint_graph_v2")
os.makedirs(OUT_DIR, exist_ok=True)

CLUSTER_KEY = 'hc_ward_k10'

# ── Load ─────────────────────────────────────────────────────────────
print("Loading adata...")
adata = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)
print("Loaded:", adata.shape)

X = adata.X.toarray() if sp_sparse.issparse(adata.X) else adata.X

# ── Compositional graph ───────────────────────────────────────────────
print("\nBuilding compositional graph...")
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=10)
A_comp = adata.obsp['connectivities'].copy()
A_comp_norm = A_comp / A_comp.max()
print("Compositional graph built")

# ── Per-slide spatial graph ───────────────────────────────────────────
def build_per_slide_spatial_graph(adata, n_neighbors=15):
    """Build spatial KNN graph independently per slide then combine."""
    n_spots = adata.n_obs
    row_indices = []
    col_indices = []
    data_values = []
    slides = adata.obs['slide_id'].unique()
    print(f"Building spatial graph for {len(slides)} slides...")

    for slide in slides:
        mask = adata.obs['slide_id'] == slide
        idx = np.where(mask)[0]

        if len(idx) < n_neighbors + 1:
            print(f"  Skipping {slide} — too few spots ({len(idx)})")
            continue

        coords_slide = adata.obsm['spatial'][idx]
        A_slide = kneighbors_graph(coords_slide,
                                    n_neighbors=n_neighbors,
                                    mode='connectivity',
                                    include_self=False)
        A_slide = (A_slide + A_slide.T) / 2  # symmetrise

        rows_local, cols_local = A_slide.nonzero()
        rows_global = idx[rows_local]
        cols_global = idx[cols_local]

        row_indices.extend(rows_global.tolist())
        col_indices.extend(cols_global.tolist())
        data_values.extend(A_slide.data.tolist())

    A_spat_full = sp_sparse.csr_matrix(
        (data_values, (row_indices, col_indices)),
        shape=(n_spots, n_spots)
    )
    print(f"Per-slide spatial graph built: {A_spat_full.nnz} edges")
    return A_spat_full

A_spat = build_per_slide_spatial_graph(adata, n_neighbors=15)
A_spat_norm = A_spat / A_spat.max()

# ── Helper functions ──────────────────────────────────────────────────
def compute_homophily_per_slide(adata, label_key, n_neighs=6):
    scores = []
    for slide in adata.obs['slide_id'].unique():
        mask = adata.obs['slide_id'] == slide
        adata_slide = adata[mask].copy()
        if adata_slide.n_obs < 50:
            continue
        sq.gr.spatial_neighbors(adata_slide,
                                 coord_type='generic',
                                 n_neighs=n_neighs,
                                 key_added='spatial_neighbors')
        adj = adata_slide.obsp['spatial_neighbors_connectivities']
        src, tgt = adj.nonzero()
        labels = adata_slide.obs[label_key].values
        same = (labels[src] == labels[tgt]).sum()
        scores.append(same / len(src))
    return np.mean(scores), np.std(scores)

def compute_entropy(adata, label_key, groupby='slide_id'):
    n_groups = adata.obs[groupby].nunique()
    H_max = np.log2(n_groups) if n_groups > 1 else 1
    per_cluster = []
    weights = []
    for cluster in adata.obs[label_key].unique():
        mask = adata.obs[label_key] == cluster
        counts = adata.obs.loc[mask, groupby].value_counts()
        props = counts / counts.sum()
        H = scipy_entropy(props, base=2)
        per_cluster.append(H / H_max)
        weights.append(mask.sum())
    return np.average(per_cluster, weights=weights)

# ── Alpha sweep ───────────────────────────────────────────────────────
alphas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
resolution = 0.5

np.random.seed(42)
n_sample = min(15000, adata.n_obs)
idx_sil = np.random.choice(adata.n_obs, n_sample, replace=False)
X_sil = X[idx_sil]

results = []
print("\nAlpha sweep with all metrics...")
print("="*60)

for alpha in alphas:
    print(f"\nalpha={alpha:.1f}...")
    G_joint = (alpha * A_comp_norm) + ((1 - alpha) * A_spat_norm)

    key = f'joint_alpha_{alpha}'
    sc.tl.leiden(adata, resolution=resolution,
                 adjacency=G_joint, key_added=key,
                 flavor='igraph', n_iterations=2,
                 directed=False, random_state=42)

    n_niches = adata.obs[key].nunique()
    print(f"  n_niches={n_niches}")

    # Silhouette
    labels_sil = adata.obs[key].values[idx_sil]
    sil = silhouette_score(X_sil, labels_sil) if n_niches > 1 else 0
    print(f"  silhouette={sil:.4f}")

    # Homophily per slide
    hom_mean, hom_std = compute_homophily_per_slide(adata, key)
    print(f"  homophily={hom_mean:.4f} ± {hom_std:.4f}")

    # Entropy
    ent_slide = compute_entropy(adata, key, groupby='slide_id')
    ent_patient = compute_entropy(adata, key, groupby='patient')
    print(f"  entropy_slide={ent_slide:.4f}, entropy_patient={ent_patient:.4f}")

    results.append({
        'alpha': alpha,
        'n_niches': n_niches,
        'silhouette': round(sil, 4),
        'homophily_mean': round(hom_mean, 4),
        'homophily_std': round(hom_std, 4),
        'entropy_slide': round(ent_slide, 4),
        'entropy_patient': round(ent_patient, 4),
    })

results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("ALPHA SWEEP COMPLETE")
print("="*60)
print(results_df.to_string(index=False))
results_df.to_csv(os.path.join(OUT_DIR, 'alpha_sweep_results.csv'), index=False)

# ── Plot all four metrics ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(results_df['alpha'], results_df['silhouette'],
                'o-', color='steelblue', linewidth=2)
axes[0, 0].set_xlabel('Alpha')
axes[0, 0].set_ylabel('Silhouette Score')
axes[0, 0].set_title('Compositional compactness\n'
                      '(biased toward α=1.0 — use with caution)')

axes[0, 1].plot(results_df['alpha'], results_df['homophily_mean'],
                'o-', color='darkorange', linewidth=2)
axes[0, 1].fill_between(
    results_df['alpha'],
    results_df['homophily_mean'] - results_df['homophily_std'],
    results_df['homophily_mean'] + results_df['homophily_std'],
    alpha=0.2, color='darkorange'
)
axes[0, 1].set_xlabel('Alpha')
axes[0, 1].set_ylabel('Spatial Homophily (mean ± std)')
axes[0, 1].set_title('Spatial coherence\n'
                      '(key metric — look for plateau)')

axes[1, 0].plot(results_df['alpha'], results_df['entropy_slide'],
                'o-', color='forestgreen', linewidth=2, label='By slide')
axes[1, 0].plot(results_df['alpha'], results_df['entropy_patient'],
                's--', color='darkgreen', linewidth=2, label='By patient')
axes[1, 0].axhline(y=0.7, color='red', linestyle='--',
                    alpha=0.7, label='Threshold (0.7)')
axes[1, 0].set_xlabel('Alpha')
axes[1, 0].set_ylabel('Normalised Entropy')
axes[1, 0].set_title('Cross-slide generalisation\n'
                      '(should stay above 0.7)')
axes[1, 0].set_ylim(0, 1)
axes[1, 0].legend()

axes[1, 1].plot(results_df['alpha'], results_df['n_niches'],
                'o-', color='purple', linewidth=2)
axes[1, 1].set_xlabel('Alpha')
axes[1, 1].set_ylabel('Number of niches')
axes[1, 1].set_title('Niche count stability')

plt.suptitle(
    'Joint graph alpha sweep — all metrics\n'
    'α=1.0 = pure composition | α=0.0 = pure spatial',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'alpha_sweep_all_metrics.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: alpha_sweep_all_metrics.png")

# ── Comparison table: joint graph vs HC Ward k=10 ────────────────────
print("\nNMI comparison: each alpha vs HC Ward k=10...")
nmi_rows = []
for alpha in alphas:
    key = f'joint_alpha_{alpha}'
    nmi = normalized_mutual_info_score(
        adata.obs[CLUSTER_KEY].astype(str),
        adata.obs[key].astype(str)
    )
    nmi_rows.append({'alpha': alpha, 'NMI_vs_HCWard_k10': round(nmi, 4)})
    print(f"  alpha={alpha:.1f}: NMI={nmi:.4f}")

nmi_df = pd.DataFrame(nmi_rows)
results_df = results_df.merge(nmi_df, on='alpha')
results_df.to_csv(os.path.join(OUT_DIR, 'alpha_sweep_results.csv'), index=False)
print("\nUpdated results saved with NMI column.")

# ── Save all alpha columns to h5ad ───────────────────────────────────
# Save all — do NOT auto-select based on silhouette
# Alpha will be chosen visually after H&E overlay inspection
print("\nSaving all joint_alpha_* columns to h5ad...")
adata.write_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)
print("Done. Next step: plot H&E overlays for each alpha and select visually.")
print("\nFinal results table:")
print(results_df.to_string(index=False))

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import scipy.sparse as sp
import os

# ── Paths ────────────────────────────────────────────────────────────
OUT_DIR = ("/path/to/data/"
           "Cell2location/spatial_mapping/exploration_mean/"
           "niche_biology")
os.makedirs(OUT_DIR, exist_ok=True)

CLUSTER_KEY = 'joint_alpha_0.9'
CELL_TYPE_ORDER = [                                                                                                                                    
      'epithelial_luminal', 'epithelial_basal', 'epithelial_club',                                                                                       
      'epithelial_hillock', 'epithelial_neuroendocrine',                                                                                                 
      'smooth_muscle', 'fibroblast', 'endothelial',                                                                                                      
      'macrophage_group_1', 'macrophage_group_2', 'monocyte', 'dendritic', 'mast',                                                                       
      'natural_killer_cd56_bright', 'natural_killer_cd56_dim',
      'b', 'plasma',                                                                                                                                     
      't_cd4_naive_central_memory', 't_cd4_tissue_resident_memory',
      't_cd8_tissue_resident_memory', 't_cd8_cytotoxic', 't_regulatory',                                                                                 
  ]

# ── Load files ───────────────────────────────────────────────────────
adata_mean = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)

adata_vis = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/spatial_mapped_with_posterior.h5ad"
)

# ── Extract raw mean abundances ──────────────────────────────────────
mean_abundances = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
mean_abundances.columns = [col.replace('meanscell_abundance_w_sf_', '')
                            for col in mean_abundances.columns]
print("Mean abundances shape:", mean_abundances.shape)
print("Cell types:", mean_abundances.columns.tolist())

# ── Build proportion matrix ──────────────────────────────────────────
cluster_proportions = pd.DataFrame(
    mean_abundances.values,
    index=adata_mean.obs.index,
    columns=mean_abundances.columns
)

# Row-normalise so each spot sums to 1
cluster_proportions = cluster_proportions.div(
    cluster_proportions.sum(axis=1), axis=0)

# Add niche label
cluster_proportions['niche'] = adata_mean.obs[CLUSTER_KEY].values

# Mean proportion per niche across all spots
niche_means = cluster_proportions.groupby('niche')[mean_abundances.columns].mean()
niche_means.index = 'Niche_' + niche_means.index.astype(str)

print("\nNiche means shape:", niche_means.shape)
print(niche_means.round(3))

# ── Clustermap ───────────────────────────────────────────────────────               
g = sns.clustermap(
      niche_means.T.loc[CELL_TYPE_ORDER],
      row_cluster=False,                                                                                                                                 
      col_cluster=True,
      cmap='mako_r',                                                                                                                                     
      standard_scale=1,
      figsize=(12, 10),
      xticklabels=True,
      yticklabels=True,                                                                                                                                  
      linewidths=0.3,
      linecolor='white',                                                                                                                                 
      dendrogram_ratio=(0.0, 0.15),
      cbar_pos=(1.02, 0.3, 0.02, 0.4),
      cbar_kws={'label': 'Row-normalised proportion\n(0=min, 1=max per cell type)'}                                                                      
  )

g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(),
    rotation=45, ha='right', fontsize=10
)
g.ax_heatmap.set_yticklabels(
    g.ax_heatmap.get_yticklabels(),
    rotation=0, fontsize=10
)

g.fig.suptitle(
    'Niche cell type profiles —joint_alpha_0.9\n'
    'Mean cell type proportion per niche (all 38 slides)',
    y=1.02, fontsize=12
)

g.savefig(os.path.join(OUT_DIR, 'niche_clustermap_joint_alpha_0.9.png'),
          dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

redo clustermap with dendogram for rows:

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import scipy.sparse as sp
import os

# ── Paths ────────────────────────────────────────────────────────────
OUT_DIR = ("/path/to/data/"
           "Cell2location/spatial_mapping/joint_graph_v2_final/"
           "niche_biology")
os.makedirs(OUT_DIR, exist_ok=True)

CLUSTER_KEY = 'joint_alpha_0.9'
CELL_TYPE_ORDER = [                                                                                                                                    
      'epithelial_luminal', 'epithelial_basal', 'epithelial_club',                                                                                       
      'epithelial_hillock', 'epithelial_neuroendocrine',                                                                                                 
      'smooth_muscle', 'fibroblast', 'endothelial',                                                                                                      
      'macrophage_group_1', 'macrophage_group_2', 'monocyte', 'dendritic', 'mast',                                                                       
      'natural_killer_cd56_bright', 'natural_killer_cd56_dim',
      'b', 'plasma',                                                                                                                                     
      't_cd4_naive_central_memory', 't_cd4_tissue_resident_memory',
      't_cd8_tissue_resident_memory', 't_cd8_cytotoxic', 't_regulatory',                                                                                 
  ]

# ── Load files ───────────────────────────────────────────────────────
adata_mean = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/exploration_mean/"
    "spatial_compositional_space_mean.h5ad"
)

adata_vis = sc.read_h5ad(
    "/path/to/data/"
    "Cell2location/spatial_mapping/spatial_mapped_with_posterior.h5ad"
)

# ── Extract raw mean abundances ──────────────────────────────────────
mean_abundances = adata_vis.obsm['means_cell_abundance_w_sf'].copy()
mean_abundances.columns = [col.replace('meanscell_abundance_w_sf_', '')
                            for col in mean_abundances.columns]
print("Mean abundances shape:", mean_abundances.shape)
print("Cell types:", mean_abundances.columns.tolist())

# ── Build proportion matrix ──────────────────────────────────────────
cluster_proportions = pd.DataFrame(
    mean_abundances.values,
    index=adata_mean.obs.index,
    columns=mean_abundances.columns
)

# Row-normalise so each spot sums to 1
cluster_proportions = cluster_proportions.div(
    cluster_proportions.sum(axis=1), axis=0)

# Add niche label
cluster_proportions['niche'] = adata_mean.obs[CLUSTER_KEY].values

# Mean proportion per niche across all spots
niche_means = cluster_proportions.groupby('niche')[mean_abundances.columns].mean()
niche_means.index = 'Niche_' + niche_means.index.astype(str)

print("\nNiche means shape:", niche_means.shape)
print(niche_means.round(3))

# ── Clustermap ───────────────────────────────────────────────────────               
g = sns.clustermap(
      niche_means.T,
      row_cluster=True,                                                                                                                                 
      col_cluster=True,
      cmap='mako_r',                                                                                                                                     
      standard_scale=1,
      figsize=(12, 10),
      xticklabels=True,
      yticklabels=True,                                                                                                                                  
      linewidths=0.3,
      linecolor='white',                                                                                                                                 
      dendrogram_ratio=(0.15, 0.15),
      cbar_pos=(1.02, 0.3, 0.02, 0.4),
      cbar_kws={'label': 'Row-normalised proportion\n(0=min, 1=max per cell type)'}                                                                      
  )

g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(),
    rotation=45, ha='right', fontsize=10
)
g.ax_heatmap.set_yticklabels(
    g.ax_heatmap.get_yticklabels(),
    rotation=0, fontsize=10
)

g.fig.suptitle(
    'Niche cell type profiles —joint_alpha_0.9\n'
    'Mean cell type proportion per niche (all 38 slides)',
    y=1.02, fontsize=12
)

g.savefig(os.path.join(OUT_DIR, 'niche_clustermap_joint_alpha_0.9.png'),
          dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")